# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

> **Croissant Schema URL**: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets with their @id and names
print("Available Record Sets:")
record_sets = dataset.list_record_sets()
for rs in record_sets:
    print(f"@id: {rs['@id']}, name: {rs['name']}")

# For each record set, also print the fields (columns) and their @id
for rs in record_sets:
    print(f"\nFields for Record Set '{rs['name']}' (@id: {rs['@id']}):")
    fields = dataset.list_fields(record_set=rs['@id'])
    for field in fields:
        col_info = field.get('column', [])
        # If column is a list of dicts, get their @id
        if isinstance(col_info, list):
            col_ids = [col['@id'] if isinstance(col, dict) and '@id' in col else str(col) for col in col_info]
            columns_repr = ', '.join(col_ids)
        elif isinstance(col_info, dict):
            columns_repr = col_info.get('@id', str(col_info))
        else:
            columns_repr = str(col_info)
        print(f"  Field name: {field['name']} | @id: {field['@id']} | column(s): {columns_repr}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s identified above.

In [ ]:
#---
# Example extraction for all available record sets
#---

dataframes = {}
record_set_ids = [rs['@id'] for rs in dataset.list_record_sets()]

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"\nLoaded DataFrame for Record Set: {record_set_id}")
            print(f"Columns: {df.columns.tolist()}")
            display(df.head())
        else:
            print(f"No records for Record Set {record_set_id}.")
    except Exception as e:
        print(f"Error loading records for Record Set {record_set_id}: {e}")

# For demonstration, pick the first record set for further EDA
main_record_set_id = record_set_ids[0] if record_set_ids else None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Operations include removing outliers, transforming data, or grouping by key attributes to prepare for further analysis.

In [ ]:
import numpy as np

if main_record_set_id and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    # Attempt to guess numeric fields (e.g. columns containing float or int datatypes)
    numeric_columns = [col for col in df.columns if np.issubdtype(df[col].dtype, np.number)]
    if numeric_columns:
        numeric_field = numeric_columns[0]  # for demonstration, just take first
        print(f"Using numeric field: {numeric_field}")
        threshold = df[numeric_field].mean() if df[numeric_field].dtype != object else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        )
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Attempt to group by a string/categorical field if present
        candidate_group_fields = [col for col in df.columns if df[col].dtype == object and col != numeric_field]
        if candidate_group_fields:
            group_field = candidate_group_fields[0]
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            display(grouped_df.head())
        else:
            print("No non-numeric group field found for groupby analysis.")
    else:
        print(f"No numeric fields detected in Record Set {main_record_set_id} for EDA.")
else:
    print(f"No available data for main Record Set {main_record_set_id}.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and main_record_set_id in dataframes and numeric_columns:
    df = dataframes[main_record_set_id]
    # Plot the histogram of the main numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field} in Record Set {main_record_set_id}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # If there is a group_field (categorical), plot boxplot
    if 'group_field' in locals():
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Loaded dataset and metadata from its FAIR Croissant schema using `mlcroissant`.
- Identified available record sets and field (column) identifiers using their `@id`s.
- Performed data extraction, filtering, normalization, and grouping on main data tables.
- Explored data distributions visually for the main numeric variable(s).

**Further work:**
- Explore relationships between more variables (e.g., peptides, abundances, modifications).
- Apply domain-specific filtering (e.g., select proteins with high coverage or specific molecular weights).
